# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is specified via a Croissant schema URL and contains multiple record sets, fields, and columns accessible using their `@id` fields.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata from Croissant schema
dataset = mlc.Dataset(croissant_url)

# Print summary (use __dict__ for metadata to avoid subscripting)
meta_obj = dataset.metadata
print(f"Name: {meta_obj.name}\nDescription: {meta_obj.description}")

## 2. Data Overview
Review available record sets and fields. Each is referenced by their `@id`, as required for reproducibility and downstream work.

In [ ]:
# List all available record sets and their fields
print('Available Record Sets:')
for rset in dataset.metadata.record_sets:
    print(f'  RecordSet @id: {rset.id} | name: {getattr(rset, "name", "-") or "-"}')
    if hasattr(rset, 'fields'):
        for field in rset.fields:
            print(f'    Field @id: {field.id} | name: {getattr(field, "name", "-") or "-"} | dataType: {getattr(field, "data_type", "-") or "-"}')
    print('-'*60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Identify all RecordSet @ids in the dataset
record_set_ids = [rset.id for rset in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records_gen = dataset.records(record_set=record_set_id)
        df = pd.DataFrame(records_gen)
        dataframes[record_set_id] = df
        print(f'Successfully loaded RecordSet {record_set_id} with columns: {df.columns.tolist()}')
        print(df.head())
    except Exception as e:
        print(f'Could not load record set {record_set_id}:', e)

# Select a representative record set for further analysis
record_set_selection = record_set_ids[0] if len(record_set_ids) > 0 else None  # Pick the first
if record_set_selection:
    print(f'Proceeding with RecordSet @id: {record_set_selection}')
    print('Columns:', dataframes[record_set_selection].columns.tolist())
    display(dataframes[record_set_selection].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records, normalizing numeric fields, and grouping. Use `@id` for fields.

In [ ]:
# For demonstration, select a numeric field from the chosen RecordSet
# (Here, select the first numeric field in that record set)
numeric_field_id = None
group_field_id = None
df = dataframes[record_set_selection].copy() if record_set_selection else pd.DataFrame()

# Find a numeric field
for rset in dataset.metadata.record_sets:
    if rset.id == record_set_selection:
        for field in getattr(rset, 'fields', []):
            # 'data_type' can be 'Integer', 'Float', etc.
            if hasattr(field, 'data_type') and str(field.data_type).lower() in ['float', 'number', 'integer']:
                numeric_field_id = field.id  # Save first numeric field
                break
        # Pick a text or non-numeric field suitable for grouping
        for field in getattr(rset, 'fields', []):
            if hasattr(field, 'data_type') and str(field.data_type).lower() in ['text', 'string']:
                group_field_id = field.id
                break
        break

if numeric_field_id is not None and numeric_field_id in df.columns:
    print(f"Using numeric field @id: {numeric_field_id}")
    try:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()  # use mean for demonstration
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Group by field if available
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id, dropna=True)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
    except Exception as e:
        print('Error in EDA section:', e)
else:
    print("No numeric field found in the selected record set for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot of numeric field grouped by group field
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, analyze, and visualize the FAIR^2 dataset using the `mlcroissant` library. For robust analyses, always use field and record set `@id` values for references, as recommended by the Croissant specification. Continue your analysis by extending filtering, analysis, or modeling as needed based on your scientific goals.